# Transaction Categoriser — Training V2

**Dataset:** `DoDataThings/us-bank-transaction-categories-v2`  
**Model:** `distilbert-base-uncased`  
**Version:** V2 hyperparameters  

| Param | V1 | V2 |
|---|---|---|
| `learning_rate` | `2e-5` | `5e-5` |
| `num_train_epochs` | `3` | `5` |
| `weight_decay` | `0.0` | `0.01` |
| preprocessing | light clean | extra normalisation |

# Step 1 — Install dependencies

In [ ]:
%%capture
!pip install -q --upgrade \
    transformers==4.44.2 \
    datasets==2.21.0 \
    wandb==0.18.6 \
    evaluate==0.4.3 \
    accelerate==0.34.2 \
    peft==0.12.0 \
    scikit-learn==1.5.0 \
    huggingface-hub==0.24.7


# Step 2 — Load Kaggle Secrets

In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
login(token=secrets.get_secret("HF_TOKEN"))
wandb.login()
print("Secrets loaded successfully.")

# Step 3 — Imports

In [ ]:
import json
import re
import warnings
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import evaluate
import wandb

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

# Step 4 — Load raw dataset

In [ ]:
DATASET_ID        = "DoDataThings/us-bank-transaction-categories-v2"
SAMPLES_PER_CLASS = 3_000
RANDOM_SEED       = 42

print(f"Loading dataset: {DATASET_ID}")
raw_ds = load_dataset(DATASET_ID)
df = raw_ds["train"].to_pandas()

print(f"\nRaw shape      : {df.shape}")
print(f"Columns        : {df.columns.tolist()}")
print(f"Missing values :\n{df.isnull().sum()}")
print(f"Duplicate rows : {df.duplicated().sum()}")
print(f"\nClass distribution (raw):")
print(df["category"].value_counts().to_string())

# Step 5 — Clean & prepare data

In [ ]:
_REF_CODE_RE = re.compile(r'\b[A-Z0-9]{6,}\b')
_SPACES_RE   = re.compile(r'\s{2,}')
_SPECIAL_RE  = re.compile(r'[*#@&]+')

def clean_description(text: str) -> str:
    text = str(text).lower()
    text = _REF_CODE_RE.sub('', text)
    text = _SPECIAL_RE.sub(' ', text)
    text = _SPACES_RE.sub(' ', text).strip()
    return text

def preprocess_description(text: str) -> str:
    """V2 extra step: strip leading [debit]/[credit] prefix,
    standardise common merchant patterns."""
    text = clean_description(text)
    text = re.sub(r'^\[(debit|credit)\]\s*', '', text)
    text = re.sub(r'\bstore\s+\d+\b', 'store', text)
    text = re.sub(r'\bapt\s+\d+\b',   'apt',   text)
    return text.strip()

CLEAN_FN = preprocess_description
print("V2: using preprocess_description (extra normalisation)")

CLEAN_FN = preprocess_description

In [ ]:
df = df.drop_duplicates()
df = df.dropna(subset=["description", "category"])
df["description"] = df["description"].apply(CLEAN_FN)
df = df[df["description"].str.len() >= 3].reset_index(drop=True)

# Stratified subsample
df_sampled = (
    df.groupby("category", group_keys=False)
      .apply(lambda g: g.sample(n=min(SAMPLES_PER_CLASS, len(g)), random_state=RANDOM_SEED))
      .reset_index(drop=True)
)
print(f"Sampled shape: {df_sampled.shape}")
print("\nClass distribution after sampling:")
print(df_sampled["category"].value_counts().to_string())

# Encode labels
labels   = sorted(df_sampled["category"].unique().tolist())
id2label = {i: lbl for i, lbl in enumerate(labels)}
label2id = {lbl: i for i, lbl in enumerate(labels)}
df_sampled["label"] = df_sampled["category"].map(label2id)

print(f"\nNum classes : {len(id2label)}")
print(f"Labels      : {list(id2label.values())}")

# Save id2label.json
with open("id2label.json", "w") as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, indent=2)
print("\nSaved id2label.json")

# Train / val / test split (80 / 10 / 10)
train_df, temp_df = train_test_split(
    df_sampled, test_size=0.20, stratify=df_sampled["label"], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=RANDOM_SEED
)
print(f"Split — train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")

# Step 6 — Tokenise

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 64   # transaction strings are short — 64 tokens is sufficient

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["description"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def df_to_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_dict({
        "description": df["description"].tolist(),
        "label":       df["label"].tolist(),
    })
    return ds.map(tokenize, batched=True, batch_size=256)

print("Tokenising splits...")
train_dataset = df_to_hf_dataset(train_df)
val_dataset   = df_to_hf_dataset(val_df)
test_dataset  = df_to_hf_dataset(test_df)
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch",   columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch",  columns=["input_ids", "attention_mask", "label"])
print(f"Sample tokenised: {train_dataset[0]['input_ids'].shape}")

# Step 7 — Load model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded    : {MODEL_NAME}")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

# Step 8 — Metrics function

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc   = accuracy_score(labels, preds)
    f1w   = f1_score(labels, preds, average="weighted", zero_division=0)
    f1m   = f1_score(labels, preds, average="macro",    zero_division=0)
    return {"accuracy": acc, "f1_weighted": f1w, "f1_macro": f1m}

# Step 9 — Initialise W&B + TrainingArguments (V2)

In [ ]:
RUN_NAME = "run-v2"

wandb.init(
    project="mlops-transaction-classifier",
    name=RUN_NAME,
    config={
        "model":            MODEL_NAME,
        "dataset":          DATASET_ID,
        "samples_per_class": SAMPLES_PER_CLASS,
        "epochs":           5,
        "batch_size":       32,
        "learning_rate":    5e-5,
        "weight_decay":     0.01,
        "max_length":       MAX_LENGTH,
        "preprocessing":    "extra_normalisation",
        "version":          "v2",
        "platform":         "Kaggle",
    },
)

training_args = TrainingArguments(
    output_dir              = "./results-v2",
    num_train_epochs        = 5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    learning_rate           = 5e-5,
    weight_decay            = 0.01,
    eval_strategy           = "epoch",
    save_strategy           = "epoch",
    load_best_model_at_end  = True,
    metric_for_best_model   = "f1_weighted",
    greater_is_better       = True,
    report_to               = "wandb",
    run_name                = RUN_NAME,
    fp16                    = torch.cuda.is_available(),
    logging_steps           = 50,
    dataloader_num_workers  = 2,
)
print("TrainingArguments ready — V2")

# Step 10 — Train

In [ ]:
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training...")
trainer.train()
print("Training complete.")

# Step 11 — Evaluate on test set

In [ ]:
print("\n=== Test-set Evaluation ===")
test_results = trainer.predict(test_dataset)
preds  = test_results.predictions.argmax(axis=-1)
labels = test_results.label_ids

print(f"Accuracy   : {accuracy_score(labels, preds):.4f}")
print(f"F1 Weighted: {f1_score(labels, preds, average='weighted', zero_division=0):.4f}")
print(f"F1 Macro   : {f1_score(labels, preds, average='macro',    zero_division=0):.4f}")
print("\nPer-class report:")
print(classification_report(labels, preds,
                             target_names=list(id2label.values()),
                             zero_division=0))

# Log test metrics to W&B
wandb.run.summary["test_accuracy"]    = accuracy_score(labels, preds)
wandb.run.summary["test_f1_weighted"] = f1_score(labels, preds, average="weighted", zero_division=0)
wandb.run.summary["test_f1_macro"]    = f1_score(labels, preds, average="macro",    zero_division=0)

# Step 12 — Push best model to HuggingFace Hub

In [ ]:
HF_REPO = "fahadkamraan/transaction-categorizer"

print(f"Pushing model to {HF_REPO} ...")
trainer.model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

wandb.run.summary["huggingface_model"] = f"https://huggingface.co/{HF_REPO}"
print(f"Model pushed: https://huggingface.co/{HF_REPO}")

# Step 13 — Finish W&B run

In [ ]:
wandb.finish()
print("W&B run finished. Check your dashboard at https://wandb.ai/fahadkamraan/mlops-transaction-classifier")